# Лабораторная работа №2. Сравнение групп, корреляция и регрессия

## Контекст задачи

Вы продолжаете работать аналитиком данных в страховой компании **MedGuard** (см. лабораторную №1).

После первого отчёта продакт-менеджер (PM) знает, *кто* наши клиенты. Теперь он хочет понять, **от чего зависят выплаты** и можно ли их **прогнозировать**:

> «Мне не нужны просто средние. Скажи, какие группы клиентов действительно отличаются по выплатам, а какие различия — шум. И дай мне формулу, по которой можно прикинуть ожидаемые выплаты для нового клиента».

**Ваша задача** — сравнить группы клиентов (t-критерий, дисперсионный анализ), найти взаимосвязи (корреляция) и построить модель выплат (линейная регрессия).

---

## Датасет

Тот же **Medical Cost Personal Dataset**, что и в лабораторной №1. Файл `insurance.csv` должен лежать в одной папке с этим ноутбуком:

- https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv

| Колонка | Описание |
|---|---|
| `age` | возраст застрахованного, лет |
| `sex` | пол: `male` / `female` |
| `bmi` | индекс массы тела (кг/м²) |
| `children` | количество детей, покрытых полисом |
| `smoker` | курит ли клиент: `yes` / `no` |
| `region` | регион: `northeast`, `northwest`, `southeast`, `southwest` |
| `charges` | сумма страховых выплат за год, USD |

---

## Правила выполнения

1. Все расчёты — в коде (`numpy`, `pandas`, `scipy.stats`, `statsmodels`, `matplotlib`/`seaborn`), как в лекции.
2. Где написано **«вручную»** — считайте по формулам из лекции, а затем сверяйтесь с готовой функцией.
3. После каждого задания — **вывод своими словами**: что посчитали и что это значит для бизнеса.
4. Уровень значимости: $\alpha = 0.05$.
5. Ячейки с `# ваш код` и `**Вывод:**` заполняете вы.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid")

df = pd.read_csv("insurance.csv")
print(df.shape)
df.head()

## Задание 1. t-распределение: когда данных мало

Отдел продаж принёс выгрузку всего по **15 курильщикам** из нового региона и хочет по ней оценить среднюю выплату. Смоделируем эту ситуацию: возьмём случайную выборку из 15 курильщиков нашей базы.

**1.1.** Сделайте выборку `df[df.smoker == "yes"].charges.sample(15, random_state=3)`. Посчитайте $\bar{X}$, $sd$, $se = sd/\sqrt{n}$.

**1.2.** Постройте 95% доверительный интервал для среднего **двумя способами**: с коэффициентом нормального распределения ($1.96$) и с коэффициентом t-распределения при $df = n - 1$ (`stats.t.ppf`). Какой интервал шире и почему? Какой из них правильный в этой ситуации?

**1.3.** Начиная с какого $n$ критическое значение t-распределения (для 95%) отличается от $1.96$ меньше чем на $0.01$? Найдите циклом.

In [ ]:
# ваш код: выборка из 15 курильщиков, X_, sd, se

In [ ]:
# ваш код: 95% ДИ по z и по t

In [ ]:
# ваш код: при каком n разница t и z меньше 0.01

**Вывод:** 

## Задание 2. Сравнение двух средних: t-критерий Стьюдента

Финансовый отдел хочет ввести для курильщиков отдельный тариф. PM просит доказать, что разница в выплатах — не случайность. Заодно маркетинг утверждает, что «мужчины обходятся дороже женщин».

**2.1.** Для групп `smoker = yes / no` соберите таблицу: `Mx`, `SD`, `N`, `SE` и половину 95% ДИ (как в лекции, задача 1 раздела «Сравнение двух средних»). Постройте рядом boxplot и график средних с доверительными интервалами (`plt.errorbar`).

**2.2.** **Вручную** посчитайте t-критерий:
$$ t = \frac{\bar{X_1} - \bar{X_2}}{\sqrt{\frac{sd_1^2}{n_1} + \frac{sd_2^2}{n_2}}}, \qquad df = n_1 + n_2 - 2 $$
и p-значение (`stats.t.sf`, двусторонняя альтернатива). Сформулируйте $H_0$ и $H_1$.

**2.3.** Проверьте результат функцией `stats.ttest_ind(..., equal_var=False)`.

**2.4.** Проверьте утверждение маркетинга: отличаются ли средние выплаты мужчин и женщин? Затем повторите тот же тест **только среди некурящих**. Как изменился вывод? Посмотрите на долю курильщиков среди мужчин и женщин и объясните, что произошло (вспомните из лекции про «третью переменную»).

In [ ]:
# ваш код: таблица Mx, SD, N, SE, interval по группам smoker; boxplot и errorbar

In [ ]:
# ваш код: t-критерий вручную

In [ ]:
# ваш код: проверка через stats.ttest_ind

In [ ]:
# ваш код: мужчины vs женщины — все клиенты и только некурящие; доля курильщиков по полу

**Вывод:** 

## Задание 3. Однофакторный дисперсионный анализ: различаются ли регионы?

Руководитель региона `southeast` снова просит увеличить бюджет: «у нас выплаты выше, чем у всех». t-критерий сравнивает только две группы, а регионов четыре — нужен **ANOVA**.

**3.1.** Посчитайте средние `charges` по регионам и постройте boxplot `charges` по `region`.

**3.2.** **Вручную** по формулам из лекции посчитайте $SS_{total}$, $SS_{between}$, $SS_{within}$, степени свободы, $MS_{bg}$, $MS_{wg}$, F-значение и p-значение (`stats.f.sf`). Убедитесь, что $SS_{total} = SS_{between} + SS_{within}$.

**3.3.** Проверьте результат функцией `stats.f_oneway`. Оформите результат в таблицу как в лекции (`Df`, `Sum Sq`, `Mean Sq`, `F-value`, `p-value`).

**3.4.** Отвечает ли ANOVA на вопрос «**какой именно** регион отличается»? Что ещё стоило бы сделать?

**3.5.** Повторите ANOVA для `bmi` по регионам (можно сразу `stats.f_oneway`). Как это может объяснить различия в выплатах?

In [ ]:
# ваш код: средние по регионам и boxplot

In [ ]:
# ваш код: SST, SSB, SSW, df, MS, F, p вручную

In [ ]:
# ваш код: stats.f_oneway и таблица результатов

In [ ]:
# ваш код: ANOVA для bmi по регионам

**Вывод:** 

## Задание 4. Корреляция: что связано с выплатами?

**4.1.** Постройте матрицу корреляций Пирсона для `age`, `bmi`, `children`, `charges` и нарисуйте её как тепловую карту (`sns.heatmap(..., annot=True)`). Какая переменная сильнее всего связана с `charges`?

**4.2.** **Вручную** посчитайте ковариацию и коэффициент корреляции между `age` и `charges`:
$$ cov(X, Y) = \frac{\sum (x_i - \bar{X})(y_i - \bar{Y})}{N - 1}, \qquad r = \frac{cov(X, Y)}{sd_X \, sd_Y} $$
Сверьтесь со `stats.pearsonr` (там же — p-значение). Посчитайте коэффициент детерминации $R^2 = r^2$ и объясните его смысл.

**4.3.** Постройте диаграмму рассеяния `age` → `charges`, раскрасив точки по `smoker` (`sns.scatterplot(..., hue="smoker")`). Что вы видите? Посчитайте $r$ между `age` и `charges` **только для некурящих** и сравните с общим. Какое условие применения коэффициента Пирсона здесь нарушено?

In [ ]:
# ваш код: матрица корреляций и heatmap

In [ ]:
# ваш код: ковариация и корреляция age–charges вручную, pearsonr, R^2

In [ ]:
# ваш код: scatter age–charges по smoker; r только для некурящих

**Вывод:** 

## Задание 5. Регрессия с одной независимой переменной

PM просит «формулу»: сколько в среднем добавляет к выплатам каждый год возраста?

**5.1.** **Вручную** по формулам МНК из лекции найдите коэффициенты прямой $\hat{y} = b_0 + b_1 x$ для `charges ~ age`:
$$ b_1 = \frac{sd_y}{sd_x} r_{xy}, \qquad b_0 = \bar{Y} - b_1 \bar{X} $$

**5.2.** Постройте модель `ols("charges ~ age", data=df).fit()` и выведите `summary()`. Совпадают ли коэффициенты? Проинтерпретируйте $b_1$, его p-значение и $R^2$.

**5.3.** Какую выплату модель предсказывает для клиента 50 лет?

**5.4.** Проверьте условия применения регрессии: постройте график остатков от `age` и QQ-plot остатков (`sm.qqplot(..., line="s")`). Выполняются ли требования нормальности остатков и гомоскедастичности?

In [ ]:
# ваш код: b0 и b1 вручную

In [ ]:
# ваш код: модель ols и summary

In [ ]:
# ваш код: предсказание для 50 лет

In [ ]:
# ваш код: график остатков и QQ-plot

**Вывод:** 

## Задание 6. Множественная регрессия и выбор лучшей модели

**6.1.** Постройте несколько моделей и сравните их по **исправленному** $R^2$ (`rsquared_adj`):
- `charges ~ age`
- `charges ~ age + bmi`
- `charges ~ age + bmi + smoker`
- `charges ~ age + bmi + smoker + children`
- `charges ~ age + bmi + smoker + children + sex + region`

Соберите результат в одну таблицу. Почему сравнивать нужно именно исправленный $R^2$, а не обычный?

**6.2.** Выберите лучшую модель с точки зрения «качество / простота». Выведите её `summary()` и проинтерпретируйте коэффициенты, особенно при `smoker` (не забудьте: «при прочих равных»).

**6.3.** В полной модели (с `sex` и `region`) посмотрите на p-значения для `sex` и `region`. Сравните с выводами заданий 2.4 и 3: подтвердилось ли, что «пол» и «регион» сами по себе важны?

**6.4.** По выбранной модели предскажите выплату для двух клиентов 50 лет с `bmi = 30`: курящего и некурящего.

In [ ]:
# ваш код: сравнение моделей по R^2 и исправленному R^2

In [ ]:
# ваш код: summary выбранной модели

In [ ]:
# ваш код: p-значения в полной модели

In [ ]:
# ваш код: прогноз для курящего и некурящего клиента

**Вывод:**

---
## Критерии оценки

| Задание | Баллы |
|---|---|
| 1. t-распределение и ДИ при малой выборке | 10 |
| 2. t-критерий Стьюдента, третья переменная | 20 |
| 3. Однофакторный дисперсионный анализ | 20 |
| 4. Корреляция | 15 |
| 5. Регрессия с одной переменной | 15 |
| 6. Множественная регрессия, выбор модели | 15 |
| 7. Отчёт для PM | 5 |
| **Итого** | **100** |

Каждое задание оценивается по трём составляющим: корректность расчёта, корректность интерпретации, качество вывода для бизнеса.